In [4]:
!pip install fastapi
!pip install uvicorn
!pip install sentence-transformers
!pip install faiss-cpu
!pip install pymupdf
!pip install mysql-connector-python
!pip install cachetools
!pip install python-multipart
!pip install numpy
!pip install python-dotenv
!pip install tqdm

In [5]:
#lib import & db config


import os
import uuid
import shutil
import traceback
import numpy as np
import fitz
import faiss
import mysql.connector

from cachetools import TTLCache
from mysql.connector import pooling
from concurrent.futures import ThreadPoolExecutor

from sentence_transformers import SentenceTransformer

from fastapi import (
    FastAPI,
    UploadFile,
    File,
    HTTPException,
    BackgroundTasks
)

from pydantic import BaseModel
from datetime import datetime
from typing import Optional


UPLOAD_DIR="uploads"

os.makedirs(
    UPLOAD_DIR,
    exist_ok=True
)

EMBEDDING_MODEL="all-MiniLM-L6-v2"

VECTOR_DIM=384

CHUNK_SIZE=500

CHUNK_OVERLAP=100

CACHE_TTL=900

MAX_WORKERS=4

TOP_K=5


DB_CONFIG={

    "host":"localhost",
    "user":"root",
    "password":"",
    "database":"knowledge_platform",
    "port":3306

}

c:\Users\Dhaval Solanki\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The file content is extracted, split into smaller chunks, converted into embeddings, and stored in MySQL(XAMPP) and FAISS.

I used MySQL through XAMPP to store document information, chunk data, and query logs. Metadata includes filename, upload status, timestamps, and deletion flags. Database storage helps tracking and managing uploaded knowledge.



In [ ]:
#db + pool

connection_pool=pooling.MySQLConnectionPool(

    pool_name="knowledge_pool",
    pool_size=10,
    **DB_CONFIG

)

def get_db():

    return connection_pool.get_connection()


def init_db():
    conn=get_db()
    cur=conn.cursor()
    cur.execute("""

    CREATE TABLE IF NOT EXISTS documents(

        id VARCHAR(255) PRIMARY KEY,
        filename TEXT,
        metadata JSON,
        status VARCHAR(50),
        created_at DATETIME,
        deleted BOOLEAN DEFAULT FALSE
    )
    """)

    cur.execute("""

    CREATE TABLE IF NOT EXISTS chunks(
        id VARCHAR(255) PRIMARY KEY,
        document_id VARCHAR(255),
        content LONGTEXT,
        chunk_order INT,
        embedding_index INT,
                
        FOREIGN KEY(document_id)
        REFERENCES documents(id)
        ON DELETE CASCADE

    )

    """)

    cur.execute("""

    CREATE TABLE IF NOT EXISTS query_logs(
        id VARCHAR(255),
        query_text LONGTEXT,
        latency FLOAT,
        results_found INT,
        created_at DATETIME
    )
    """)

    conn.commit()
    cur.close()
    conn.close()

init_db()

Documents Table: Stores document ID, filename, metadata, processing status, and delete status. Helps manage uploaded files.

Chunks Table: Stores chunk text and embedding references.Connects chunks back to original documents.

Query Logs Table: Stores user queries, latency, and result counts. Used for monitoring performance.

In [ ]:
# model + vector store

model=SentenceTransformer(
    EMBEDDING_MODEL

)
vector_store=faiss.IndexFlatIP(
    VECTOR_DIM

)
vector_map=[]
query_cache=TTLCache(
    maxsize=1000,
    ttl=CACHE_TTL
)

executor=ThreadPoolExecutor(
    max_workers=MAX_WORKERS
)

Large documents are divided into 500-character chunks with 100-character overlap. Overlap keeps context between chunks so information is not lost. Chunking improves semantic search accuracy.

I used SentenceTransformer (all-MiniLM-L6-v2) to convert text chunks into vector embeddings. Embeddings convert text meaning into numbers for AI search. Similar meaning documents stay closer in vector space.

In [ ]:
#chunking & embeddings

def extract_pdf(path):
    doc=fitz.open(path)
    text=[]
    for page in doc:
        text.append(
            page.get_text()
        )

    return "\n".join(text)

def extract_text(path):
    with open(
        path,
        encoding="utf8",
        errors="ignore"
    ) as f:
        return f.read()

def extract_document(path):
    ext=os.path.splitext(
        path
    )[1].lower()

    if ext==".pdf":
        return extract_pdf(path)
    return extract_text(path)

def chunk_document(text):
    chunks=[]
    start=0

    while start<len(text):
        end=start+CHUNK_SIZE
        chunk=text[start:end]

        chunks.append(chunk)
        start+=(

            CHUNK_SIZE-
            CHUNK_OVERLAP
        )
    return chunks

def generate_embeddings(texts):

    embeddings=model.encode(
        texts,
        convert_to_numpy=True
    )

    norm=np.linalg.norm(
        embeddings,
        axis=1,
        keepdims=True

    )

    return embeddings/norm

def add_vectors(
    vectors,
    metadata

):
    start=vector_store.ntotal

    vector_store.add(

        vectors.astype(
            "float32"
        )
    )

    for i,item in enumerate(metadata):
        item["vector_index"]=start+i
        vector_map.append(item)

I used MySQL connection pooling to reuse database connections. This reduces database overhead. Supports around 100 internal developers efficiently. 


File processing runs in background tasks. Users receive upload confirmation immediately. Heavy operations like embedding generation happen asynchronously.

In [ ]:
#processing pipeline

def process_document(

    doc_id,
    path,
    filename
):
    conn=None
    try:
        conn=get_db()
        cur=conn.cursor()
        text=extract_document(path)
        chunks=chunk_document(text)
        embeddings=generate_embeddings(
            chunks
        )
        metadata=[]
        for idx,chunk in enumerate(chunks):
            chunk_id=str(uuid.uuid4())
            metadata.append({
                "chunk_id":chunk_id,
                "doc_id":doc_id,
                "filename":filename,
                "content":chunk
            })
        add_vectors(
            embeddings,
            metadata
        )

        for idx,item in enumerate(metadata):
            cur.execute("""
            INSERT INTO chunks(
            id,
            document_id,
            content,
            chunk_order,
            embedding_index

            )
            VALUES(

            %s,%s,%s,%s,%s

            )
            """,(
            item["chunk_id"],
            doc_id,
            item["content"],
            idx,
            item["vector_index"]
            ))

        cur.execute("""
        UPDATE documents
        SET status='READY'
        WHERE id=%s
        """,(doc_id,))
        conn.commit()

    except:
        traceback.print_exc()
    finally:

        if conn:

            conn.close()

User sends a natural language question. The query is converted into embeddings and searched in FAISS. Top matching chunks are returned based on semantic similarity.


I generate embeddings for user queries. FAISS calculates vector similarity between query and document chunks.
Top-K relevant chunks are returned.

In [ ]:
#FAST API

app=FastAPI(

title="AI Knowledge Platform"

)
class QueryRequest(BaseModel):
    query:str
    top_k:int=5
    filename:Optional[str]=None

@app.post("/documents")
async def upload(

background_tasks:BackgroundTasks,

file:UploadFile=File(...)

):
    doc_id=str(uuid.uuid4())
    path=os.path.join(
        UPLOAD_DIR,
        f"{doc_id}_{file.filename}"

    )
    with open(path,"wb") as f:

        shutil.copyfileobj(
            file.file,
            f
        )
    conn=get_db()
    cur=conn.cursor()
    cur.execute("""
    INSERT INTO documents
    VALUES(
    %s,%s,%s,%s,%s,%s
    )
    """,(
    doc_id,
    file.filename,
    "{}",
    "PROCESSING",
    datetime.utcnow(),
    False

    ))

    conn.commit()

    conn.close()

    background_tasks.add_task(

        process_document,

        doc_id,
        path,
        file.filename

    )
    return {

        "id":doc_id

    }


@app.post("/query")
async def query(

request:QueryRequest

):

    start=datetime.utcnow()
    emb=generate_embeddings(
        [request.query]

    )
    scores,ids=vector_store.search(
        emb.astype("float32"),
        request.top_k*3

    )

    results=[]
    for score,idx in zip(
        scores[0],
        ids[0]

    ):

        if idx==-1:
            continue
        item=vector_map[idx]
        if request.filename:
            if item["filename"]!=request.filename:
                continue
        results.append({

            "score":float(score),
            "filename":item["filename"],
            "chunk":item["content"]
        })

        if len(results)>=request.top_k:

            break

    latency=(

        datetime.utcnow()-start

    ).total_seconds()

    conn=get_db()

    cur=conn.cursor()

    cur.execute("""

    INSERT INTO query_logs
    VALUES(
    %s,%s,%s,%s,%s
    )
    """,(
    str(uuid.uuid4()),
    request.query,
    latency,
    len(results),
    datetime.utcnow()
    ))

    conn.commit()

    conn.close()

    return results


@app.delete("/documents/{doc_id}")
async def delete(

doc_id:str,

hard_delete:bool=False

):

    conn=get_db()

    cur=conn.cursor()

    try:

        if hard_delete:

            cur.execute(

            """

            DELETE FROM chunks
            WHERE document_id=%s
            """,

            (doc_id,)
            )
            cur.execute(

            """
            DELETE FROM documents
            WHERE id=%s
            """,

            (doc_id,)
            )

        else:
            cur.execute(
            """
            UPDATE documents
            SET deleted=TRUE
            WHERE id=%s
            """,

            (doc_id,)
            )

        conn.commit()

        return {
            "success":True
        }
    finally:
        conn.close()

@app.get("/health")
async def health():

    return {
        "vectors":
        vector_store.ntotal,

        "cache":
        len(query_cache)

    }

In [ ]:
!pip install nest_asyncio

In [ ]:
import nest_asyncio
import uvicorn
from uvicorn import Config, Server

nest_asyncio.apply()

config = Config(
    app=app,
    host="0.0.0.0",
    port=8000
)

server = Server(config)

await server.serve()

INFO:     Started server process [9052]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:62727 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:62727 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:62728 - "POST /documents HTTP/1.1" 200 OK


C:\Users\Dhaval Solanki\AppData\Local\Temp\ipykernel_9052\4043509601.py:67: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow(),


INFO:     127.0.0.1:62763 - "DELETE /documents/054f6e87-878c-4d93-a80e-1bc4b9ce36e7?hard_delete=false HTTP/1.1" 200 OK
INFO:     127.0.0.1:62765 - "DELETE /documents/054f6e87-878c-4d93-a80e-1bc4b9ce36e7?hard_delete=false HTTP/1.1" 200 OK
INFO:     127.0.0.1:62769 - "DELETE /documents/054f6e87-878c-4d93-a80e-1bc4b9ce36e7?hard_delete=true HTTP/1.1" 200 OK
INFO:     127.0.0.1:63072 - "DELETE /documents/http%3A//localhost/phpmyadmin/index.php%3Froute%3D/sql%26db%3Dknowledge_platform%26table%3Ddocuments%26pos%3D0%26sql_signature%3D334f82857d1f039562e26fa3592669e1fe315fd0ed6178be55f907ed03bb8467%26sql_query%3DSELECT%2B%252A%2BFROM%2B%2560knowledge_platform%2560.%2560documents%2560%2BWHERE%2B%2560id%2560%2B%253D%2B%2527e3bc49c0-d319-4b1a-93f0-9eaf5e8e3842%2527?hard_delete=true HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:63074 - "DELETE /documents/http%3A//localhost/phpmyadmin/index.php%3Froute%3D/sql%26db%3Dknowledge_platform%26table%3Ddocuments%26pos%3D0%26sql_signature%3D334f82857d1f039562e

C:\Users\Dhaval Solanki\AppData\Local\Temp\ipykernel_9052\4043509601.py:100: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start=datetime.utcnow()


INFO:     127.0.0.1:63167 - "POST /query HTTP/1.1" 200 OK


C:\Users\Dhaval Solanki\AppData\Local\Temp\ipykernel_9052\4043509601.py:153: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow()-start
C:\Users\Dhaval Solanki\AppData\Local\Temp\ipykernel_9052\4043509601.py:181: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow()


INFO:     127.0.0.1:63167 - "POST /query HTTP/1.1" 200 OK
INFO:     127.0.0.1:63196 - "DELETE /documents/e3bc49c0-d319-4b1a-93f0-9eaf5e8e3842?hard_delete=false HTTP/1.1" 200 OK
INFO:     127.0.0.1:63199 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:63199 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:63269 - "POST /query HTTP/1.1" 200 OK
INFO:     127.0.0.1:63292 - "POST /query HTTP/1.1" 200 OK
INFO:     127.0.0.1:63301 - "POST /query HTTP/1.1" 200 OK
INFO:     127.0.0.1:63383 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:63383 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:63384 - "POST /documents HTTP/1.1" 200 OK
INFO:     127.0.0.1:63483 - "POST /query HTTP/1.1" 200 OK
INFO:     127.0.0.1:63483 - "POST /query HTTP/1.1" 200 OK
INFO:     127.0.0.1:63492 - "DELETE /documents/e3bc49c0-d319-4b1a-93f0-9eaf5e8e3842?hard_delete=true HTTP/1.1" 200 OK
